# congress-data: Pipeline Prototyping

Run individual pipeline stages manually for testing and development.

```bash
pip install -e libs/dagster-io -e packages/congress-data
```

In [ ]:
import os
# os.environ["LLM_BASE_URL"] = "http://litellm.your-cluster:4000/v1"
# os.environ["LLM_API_KEY"] = "sk-..."

## 1. Create Sample Documents

In [ ]:
from congress_data.core.document import Document

sample_docs = [
    Document(
        id="congress-bill-hr1",
        title="Infrastructure Investment and Jobs Act",
        content=(
            "The Infrastructure Investment and Jobs Act allocates $1.2 trillion for "
            "infrastructure improvements including $110 billion for roads and bridges, "
            "$66 billion for passenger and freight rail, $65 billion for broadband expansion, "
            "and $55 billion for clean water infrastructure. The bill was signed into law "
            "by President Biden on November 15, 2021. Senator Rob Portman and Senator "
            "Kyrsten Sinema led the bipartisan negotiations. The Department of Transportation "
            "oversees the majority of the allocated funds."
        ),
        source="congress.gov",
        document_type="bill",
        domain="congress",
        entity_type="Bill",
        metadata={"congress": 117, "bill_type": "hr"},
    ),
    Document(
        id="congress-bill-hr2",
        title="CHIPS and Science Act",
        content=(
            "The CHIPS and Science Act provides $52.7 billion in subsidies for semiconductor "
            "manufacturing in the United States. It includes $39 billion in manufacturing "
            "incentives and $13.2 billion for R&D and workforce development. Intel, TSMC, "
            "and Samsung have announced major factory investments in response. The Commerce "
            "Department administers the CHIPS fund. The act was signed August 9, 2022."
        ),
        source="congress.gov",
        document_type="bill",
        domain="congress",
        entity_type="Bill",
        metadata={"congress": 117, "bill_type": "hr"},
    ),
]

print(f"Created {len(sample_docs)} sample documents")
for doc in sample_docs:
    print(f"  {doc.id}: {doc.title} ({len(doc.content)} chars)")

## 2. Chunking Stage

In [ ]:
from dagster_io import chunk_document

all_chunks = []
for doc in sample_docs:
    chunks = chunk_document(
        document_id=doc.id,
        title=doc.title,
        content=doc.content,
        metadata={"source": doc.source, "document_type": doc.document_type},
        chunk_size=300,
        chunk_overlap=50,
    )
    all_chunks.extend(chunks)

print(f"\n{len(sample_docs)} docs → {len(all_chunks)} chunks")
for chunk in all_chunks:
    print(f"\n{chunk.chunk_id} ({len(chunk.text)} chars):")
    print(f"  {chunk.text[:100]}...")

## 3. NER Extraction (requires LLM)

In [ ]:
from dagster_io import LLMResource
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

llm = LLMResource()
llm.setup_for_execution(None)

class Entity(BaseModel):
    text: str = Field(description="Entity mention")
    label: str = Field(description="PERSON, ORG, GPE, DATE, LAW, EVENT")
    context: str = Field(description="Sentence fragment")

class NERResult(BaseModel):
    entities: list[Entity]

chain = llm.with_structured_output(NERResult)

# Extract from first chunk
chunk = all_chunks[0]
result = chain.invoke([
    SystemMessage(content="Extract all named entities from this text."),
    HumanMessage(content=chunk.text),
])

print(f"Entities from {chunk.chunk_id}:")
for ent in result.entities:
    print(f"  [{ent.label}] {ent.text}")
    print(f"    context: {ent.context[:80]}")

## 4. Proposition Extraction (requires LLM)

In [ ]:
class Proposition(BaseModel):
    subject: str
    predicate: str
    object: str
    confidence: float = Field(ge=0, le=1)

class PropositionResult(BaseModel):
    propositions: list[Proposition]

spo_chain = llm.with_structured_output(PropositionResult)

result = spo_chain.invoke([
    SystemMessage(content="Extract Subject-Predicate-Object triples. Focus on factual claims."),
    HumanMessage(content=chunk.text),
])

print(f"Propositions from {chunk.chunk_id}:")
for p in result.propositions:
    print(f"  ({p.subject}) --[{p.predicate}]--> ({p.object})  [{p.confidence:.0%}]")

## 5. Embedding (requires API or local model)

In [ ]:
from dagster_io import EmbeddingResource
import numpy as np

emb = EmbeddingResource()
emb.setup_for_execution(None)

texts = [c.text for c in all_chunks]
vectors = emb.embed(texts)

print(f"Embedded {len(vectors)} chunks → {len(vectors[0])}d vectors")

# Similarity matrix
vecs = np.array(vectors)
norms = np.linalg.norm(vecs, axis=1, keepdims=True)
sim_matrix = (vecs @ vecs.T) / (norms @ norms.T)

print("\nChunk similarity matrix:")
for i, c in enumerate(all_chunks):
    sims = [f"{sim_matrix[i][j]:.2f}" for j in range(len(all_chunks))]
    print(f"  {c.chunk_id}: [{', '.join(sims)}]")

## 6. Full Pipeline (Documents → Chunks → NER + Embeddings)

In [ ]:
# Run the full pipeline on all chunks
print("=== Full Pipeline Run ===")
print(f"Documents: {len(sample_docs)}")
print(f"Chunks: {len(all_chunks)}")

# NER across all chunks
all_entities = []
for chunk in all_chunks:
    result = chain.invoke([
        SystemMessage(content="Extract all named entities."),
        HumanMessage(content=chunk.text),
    ])
    for ent in result.entities:
        all_entities.append({
            **ent.model_dump(),
            "source_doc_id": chunk.document_id,
            "chunk_id": chunk.chunk_id,
        })

print(f"Entities: {len(all_entities)}")

# Show entity counts by label
from collections import Counter
label_counts = Counter(e["label"] for e in all_entities)
for label, count in label_counts.most_common():
    print(f"  {label}: {count}")